In [2]:
# train_and_eval_code_models.py
"""
Fine-tune a code model (CodeGPT/CodeGen) and a general model (GPT-2)
on the same code dataset and evaluate with CodeBLEU + exact match.

Assumptions:
 - data/train.jsonl, data/valid.jsonl, data/test.jsonl exist
 - each line is JSON: {"id":"...", "prompt":"...", "target":"..."}
"""

import os
import json
import math
import random
from dataclasses import dataclass
from typing import Dict, List

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm
import numpy as np

# CodeBLEU import (pip package)
# If import fails, install: pip install codebleu
try:
    from codebleu.bleu import calc_code_bleu
except Exception as e:
    raise ImportError("Install codebleu: pip install codebleu (see README). Error: " + str(e))

# --------------------------
# Config - edit as needed
# --------------------------
CODE_MODEL_NAME = "microsoft/CodeGPT-small-py"    # code-specialized (small)
GEN_MODEL_NAME  = "gpt2"                          # general-purpose baseline

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = "data/train.jsonl"
VALID_FILE = "data/valid.jsonl"
TEST_FILE  = "data/test.jsonl"

# training hyperparams (small defaults so it runs on modest GPUs)
EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 5e-5
MAX_LENGTH = 512             # token length cap for prompt+target
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --------------------------
# Helpers & Dataset
# --------------------------
def read_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

class CodeDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=512, mode="train"):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mode = mode

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        prompt = rec.get("prompt", "")
        target = rec.get("target", "")

        # We will concatenate prompt + target; labels mask the prompt (set to -100)
        full = prompt + target

        enc = self.tokenizer(
            full,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        # build labels: mask prompt tokens
        # find tokenized length of prompt to mask appropriately
        prompt_enc = self.tokenizer(
            prompt, truncation=True, max_length=self.max_length, return_tensors="pt", padding=False
        )
        prompt_len = prompt_enc["input_ids"].shape[1]

        labels = input_ids.clone()
        # mask the prompt part
        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "prompt_len": prompt_len,
            "raw": {"id": rec.get("id"), "prompt": prompt, "target": target},
        }

# collate
def collate_fn(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    labels = torch.stack([b["labels"] for b in batch])
    raw = [b["raw"] for b in batch]
    prompt_lens = [b["prompt_len"] for b in batch]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels, "raw": raw, "prompt_lens": prompt_lens}


# --------------------------
# Training loop (no pipeline)
# --------------------------
def train_model(model_name, save_dir):
    print(f"\n\n=== Preparing model {model_name} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # GPT2-style models may not have pad token - set it
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.resize_token_embeddings(len(tokenizer))
    model.to(DEVICE)

    # load data
    train_recs = read_jsonl(TRAIN_FILE)
    valid_recs = read_jsonl(VALID_FILE)
    test_recs  = read_jsonl(TEST_FILE)

    train_ds = CodeDataset(train_recs, tokenizer, max_length=MAX_LENGTH, mode="train")
    valid_ds = CodeDataset(valid_recs, tokenizer, max_length=MAX_LENGTH, mode="valid")
    test_ds  = CodeDataset(test_recs, tokenizer, max_length=MAX_LENGTH, mode="test")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)  # eval generate 1-by-1

    # optimizer & scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = (len(train_loader) // GRAD_ACCUM + 1) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=math.ceil(0.06*total_steps), num_training_steps=total_steps)

    model.train()
    global_step = 0
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}/{EPOCHS}")
        pbar = tqdm(train_loader)
        running_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss = loss / GRAD_ACCUM
            loss.backward()
            running_loss += loss.item()

            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
                pbar.set_description(f"loss={running_loss/(global_step):.4f}")

        # run validation generation loss/metric after epoch
        val_loss = evaluate_loss(model, valid_loader)
        print(f"Validation loss after epoch {epoch+1}: {val_loss:.4f}")

        # save checkpoint
        ckpt_path = os.path.join(save_dir, f"checkpoint-epoch{epoch+1}")
        os.makedirs(ckpt_path, exist_ok=True)
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        print("Saved checkpoint to", ckpt_path)

    # After training evaluate on test set (generate and compute metrics)
    print("Final evaluation on test set...")
    gen_outs = generate_for_dataset(model, tokenizer, test_loader, max_new_tokens=256)
    metrics = compute_metrics_with_codebleu(gen_outs)
    print("Test metrics:", metrics)

    # save final
    final_path = os.path.join(save_dir, "final")
    os.makedirs(final_path, exist_ok=True)
    model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    # write metrics
    with open(os.path.join(final_path, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    return metrics, gen_outs

# simple evaluation of loss (no generation)
def evaluate_loss(model, dataloader):
    model.eval()
    total = 0.0
    n = 0
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total += outputs.loss.item() * input_ids.size(0)
            n += input_ids.size(0)
    model.train()
    return total / n if n > 0 else 0.0

# generation for dataset (returns list of dicts: id, prompt, target, pred)
def generate_for_dataset(model, tokenizer, dataloader, max_new_tokens=128):
    model.eval()
    results = []
    for batch in tqdm(dataloader):
        raw = batch["raw"][0]
        prompt = raw["prompt"]
        target = raw["target"]
        # encode prompt
        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).input_ids.to(DEVICE)
        # generate
        with torch.no_grad():
            gen = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,    # greedy for deterministic eval; change if desired
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id,
            )
        # decode the generated full output (includes prompt)
        decoded = tokenizer.decode(gen[0], skip_special_tokens=True)
        # extract generated portion (after prompt)
        if decoded.startswith(prompt):
            pred = decoded[len(prompt):]
        else:
            # fallback: try splitting at last newline of prompt
            pred = decoded
        results.append({"id": raw.get("id"), "prompt": prompt, "target": target, "pred": pred})
    model.train()
    return results

# --------------------------
# Metrics: CodeBLEU + exact match + token overlap (precision/recall/f1)
# --------------------------
def compute_metrics_with_codebleu(gen_outs):
    # CodeBLEU expects lists of references and candidates and a lang parameter like "python"
    # We'll use simple config: languages = "python"
    refs = [g["target"] for g in gen_outs]
    cands = [g["pred"] for g in gen_outs]

    # CodeBLEU: package calc_code_bleu(references, candidates, lang)
    # Note: the pip package expects newline-separated code or lists depending on version.
    try:
        codebleu_res = calc_code_bleu(refs, cands, "python")
    except Exception as e:
        print("CodeBLEU calculation failed:", e)
        codebleu_res = {"total_score": None, "bleu": None, "weighted_ngram_match": None, "ast_match": None, "data_flow_match": None}

    # exact match
    exact = sum(1 for r, c in zip(refs, cands) if r.strip() == c.strip()) / max(1, len(refs))

    # token-level simple precision/recall/f1 (whitespace tokenization)
    precisions = []
    recalls = []
    f1s = []
    for r, c in zip(refs, cands):
        r_tokens = r.strip().split()
        c_tokens = c.strip().split()
        if len(c_tokens) == 0 or len(r_tokens) == 0:
            precisions.append(0.0); recalls.append(0.0); f1s.append(0.0); continue
        tp = 0
        # count overlap by multiset intersection
        from collections import Counter
        rc = Counter(r_tokens)
        cc = Counter(c_tokens)
        for tok in cc:
            tp += min(cc[tok], rc.get(tok, 0))
        prec = tp / sum(cc.values()) if sum(cc.values())>0 else 0.0
        rec = tp / sum(rc.values()) if sum(rc.values())>0 else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        precisions.append(prec); recalls.append(rec); f1s.append(f1)
    token_prec = float(np.mean(precisions))
    token_rec  = float(np.mean(recalls))
    token_f1   = float(np.mean(f1s))

    metrics = {
        "codebleu": codebleu_res,
        "exact_match": exact,
        "token_precision": token_prec,
        "token_recall": token_rec,
        "token_f1": token_f1,
        "n_examples": len(refs)
    }
    return metrics

# --------------------------
# Orchestration
# --------------------------
def main():
    # Train code-specialized model
    code_save = os.path.join(OUTPUT_DIR, "code_model")
    os.makedirs(code_save, exist_ok=True)
    print("Training code-specialized model:", CODE_MODEL_NAME)
    code_metrics, code_gen_outs = train_model(CODE_MODEL_NAME, code_save)

    # Train general-purpose model
    gen_save = os.path.join(OUTPUT_DIR, "gen_model")
    os.makedirs(gen_save, exist_ok=True)
    print("Training general-purpose model:", GEN_MODEL_NAME)
    gen_metrics, gen_gen_outs = train_model(GEN_MODEL_NAME, gen_save)

    # Comparative summary
    print("\n\n=== COMPARISON SUMMARY ===")
    print("Code model ({}):".format(CODE_MODEL_NAME))
    print(json.dumps(code_metrics, indent=2))
    print("\nGeneral model ({}):".format(GEN_MODEL_NAME))
    print(json.dumps(gen_metrics, indent=2))

    # Save predictions for manual inspection
    with open(os.path.join(OUTPUT_DIR, "code_model_preds.jsonl"), "w", encoding="utf-8") as f:
        for r in code_gen_outs:
            f.write(json.dumps(r) + "\n")
    with open(os.path.join(OUTPUT_DIR, "gen_model_preds.jsonl"), "w", encoding="utf-8") as f:
        for r in gen_gen_outs:
            f.write(json.dumps(r) + "\n")

if __name__ == "__main__":
    main()

ImportError: Install codebleu: pip install codebleu (see README). Error: cannot import name 'calc_code_bleu' from 'codebleu.bleu' (/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/codebleu/bleu.py)

In [4]:
pip install sacrebleu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sacrebleu]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
Lab Exercise 7:
Fine-tune a code-specialized model and a general-purpose model,
evaluate with CodeBLEU (from CodeXGLUE) or BLEU+F1 fallback.

Data format (JSONL files in data/train.jsonl, valid.jsonl, test.jsonl):
{"id":"1","prompt":"# description or partial code\n","target":"def foo():\n    ...\n"}
"""

import os, json, math, random, sys
from collections import Counter
from typing import List, Dict

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
import numpy as np
from tqdm.auto import tqdm

# Try CodeBLEU (CodeXGLUE) first
USE_CODEBLEU = False
try:
    sys.path.append("CodeXGLUE/Code-Code/code-to-code-trans/evaluator/CodeBLEU")
    from calc_code_bleu import calc_code_bleu
    USE_CODEBLEU = True
    print("✅ CodeBLEU loaded from CodeXGLUE.")
except Exception as e:
    print("⚠️ CodeBLEU not available, will use BLEU fallback. Error:", e)
    from sacrebleu import corpus_bleu

# --------------------------
# Config
# --------------------------
CODE_MODEL_NAME = "microsoft/CodeGPT-small-py"
GEN_MODEL_NAME = "gpt2"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = "data/train.jsonl"
VALID_FILE = "data/valid.jsonl"
TEST_FILE  = "data/test.jsonl"

EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 5e-5
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --------------------------
# Dataset
# --------------------------
def read_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

class CodeDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=512):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        prompt, target = rec.get("prompt",""), rec.get("target","")
        full = prompt + target
        enc = self.tokenizer(
            full, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attn_mask = enc["attention_mask"].squeeze(0)

        # mask prompt tokens in labels
        prompt_ids = self.tokenizer(
            prompt, truncation=True, max_length=self.max_length,
            return_tensors="pt"
        )["input_ids"].squeeze(0)
        prompt_len = prompt_ids.shape[0]

        labels = input_ids.clone()
        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attn_mask,
            "labels": labels,
            "raw": {"id": rec.get("id"), "prompt": prompt, "target": target}
        }

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
        "raw": [b["raw"] for b in batch]
    }

# --------------------------
# Training
# --------------------------
def train_model(model_name, save_dir):
    print(f"\n=== Training {model_name} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.resize_token_embeddings(len(tokenizer))
    model.to(DEVICE)

    train_ds = CodeDataset(read_jsonl(TRAIN_FILE), tokenizer, MAX_LENGTH)
    valid_ds = CodeDataset(read_jsonl(VALID_FILE), tokenizer, MAX_LENGTH)
    test_ds  = CodeDataset(read_jsonl(TEST_FILE), tokenizer, MAX_LENGTH)

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    valid_loader = DataLoader(valid_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds, 1, shuffle=False, collate_fn=collate_fn)

    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = (len(train_loader)//GRAD_ACCUM + 1) * EPOCHS
    sched = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=int(0.06*total_steps), num_training_steps=total_steps
    )

    global_step, running = 0, 0
    model.train()
    for ep in range(EPOCHS):
        print(f"Epoch {ep+1}/{EPOCHS}")
        for step, batch in enumerate(tqdm(train_loader)):
            ids, mask, labels = batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE), batch["labels"].to(DEVICE)
            out = model(input_ids=ids, attention_mask=mask, labels=labels)
            loss = out.loss / GRAD_ACCUM
            loss.backward()
            running += loss.item()
            if (step+1)%GRAD_ACCUM==0 or (step+1)==len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad(); global_step+=1
                if global_step%10==0: tqdm.write(f"step {global_step} loss={running/global_step:.4f}")

    # Final test evaluation
    gen_outs = generate_for_dataset(model, tokenizer, test_loader)
    metrics = compute_metrics(gen_outs)
    print("Final test metrics:", metrics)

    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir); tokenizer.save_pretrained(save_dir)
    with open(os.path.join(save_dir,"metrics.json"),"w") as f: json.dump(metrics,f,indent=2)
    return metrics, gen_outs

# --------------------------
# Generation & Metrics
# --------------------------
def generate_for_dataset(model, tokenizer, dataloader, max_new_tokens=128):
    model.eval(); results=[]
    for batch in tqdm(dataloader):
        raw = batch["raw"][0]
        prompt, target = raw["prompt"], raw["target"]
        ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).input_ids.to(DEVICE)
        with torch.no_grad():
            gen = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id,
                                 eos_token_id=tokenizer.eos_token_id or tokenizer.sep_token_id)
        decoded = tokenizer.decode(gen[0], skip_special_tokens=True)
        pred = decoded[len(prompt):] if decoded.startswith(prompt) else decoded
        results.append({"id":raw["id"],"prompt":prompt,"target":target,"pred":pred})
    model.train(); return results

def compute_metrics(gen_outs):
    refs = [g["target"] for g in gen_outs]
    cands = [g["pred"] for g in gen_outs]

    # CodeBLEU if available
    if USE_CODEBLEU:
        try:
            res = calc_code_bleu([refs], [cands], lang="python")
            codebleu = res["codebleu"]
        except Exception as e:
            print("CodeBLEU failed, fallback BLEU:", e)
            codebleu = None
    else:
        bleu = corpus_bleu(cands, [refs]).score
        codebleu = bleu

    # exact match
    exact = sum(1 for r,c in zip(refs,cands) if r.strip()==c.strip())/len(refs)

    # token overlap F1
    f1s=[]
    for r,c in zip(refs,cands):
        rt, ct = r.split(), c.split()
        rc, cc = Counter(rt), Counter(ct)
        tp = sum(min(cc[t],rc.get(t,0)) for t in cc)
        prec = tp/sum(cc.values()) if cc else 0
        rec  = tp/sum(rc.values()) if rc else 0
        f1 = 2*prec*rec/(prec+rec) if prec+rec>0 else 0
        f1s.append(f1)
    return {"codebleu_or_bleu": codebleu, "exact_match": exact, "f1": float(np.mean(f1s))}

# --------------------------
# Main
# --------------------------
def main():
    code_metrics,_ = train_model(CODE_MODEL_NAME, os.path.join(OUTPUT_DIR,"code_model"))
    gen_metrics,_  = train_model(GEN_MODEL_NAME, os.path.join(OUTPUT_DIR,"gen_model"))

    print("\n=== COMPARISON SUMMARY ===")
    print("Code model:", code_metrics)
    print("General model:", gen_metrics)

if __name__=="__main__":
    main()


⚠️ CodeBLEU not available, will use BLEU fallback. Error: cannot import name 'ngrams' from 'utils' (/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/utils/__init__.py)

=== Training microsoft/CodeGPT-small-py ===


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import json
import requests
import os
from tqdm import tqdm
import random

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create a smaller sample dataset for demonstration
def create_sample_dataset():
    os.makedirs('data', exist_ok=True)
    
    # Sample code snippets with labels (0 = no bug, 1 = bug)
    samples = [
        {"func": "def add(a, b):\n    return a + b", "target": 0},
        {"func": "def subtract(a, b):\n    return a - b", "target": 0},
        {"func": "def divide(a, b):\n    return a / b", "target": 0},
        {"func": "def multiply(a, b):\n    return a * b", "target": 0},
        {"func": "def calculate(a, b):\n    result = a + b\n    return result", "target": 0},
        {"func": "def buggy_add(a, b):\n    return a - b  # Should be addition", "target": 1},
        {"func": "def buggy_divide(a, b):\n    return a * b  # Should be division", "target": 1},
        {"func": "def problematic_function():\n    print('Hello'  # Missing closing parenthesis", "target": 1},
        {"func": "def infinite_loop():\n    while True:\n        pass", "target": 1},
        {"func": "def null_reference():\n    x = None\n    return x.length", "target": 1},
    ]
    
    # Create train, validation, and test splits
    splits = {
        'train': samples[:6],
        'valid': samples[6:8],
        'test': samples[8:]
    }
    
    for split_name, split_data in splits.items():
        with open(f'data/{split_name}.jsonl', 'w') as f:
            for item in split_data:
                f.write(json.dumps(item) + '\n')
    
    print("Created sample dataset with 10 code examples")

# Custom Dataset class
class CodeDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = self.load_data(file_path)

    def load_data(self, file_path):
        data = []
        try:
            with open(file_path, 'r') as f:
                for line in f:
                    data.append(json.loads(line))
            print(f"Loaded {len(data)} samples from {file_path}")
        except Exception as e:
            print(f"Error loading data from {file_path}: {e}")
            return []
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        code = item['func']
        label = item['target']
        
        encoding = self.tokenizer(
            code,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Training function
def train_model(model, train_loader, val_loader, model_name, epochs=3):
    optimizer = AdamW(model.parameters(), lr=2e-5)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )

    best_accuracy = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f'{model_name} - Epoch {epoch+1}/{epochs}')
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_loss += loss.item()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            progress_bar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = total_loss / len(train_loader)
        val_accuracy, val_f1 = evaluate_model(model, val_loader, f"{model_name} Validation")
        
        print(f"{model_name} - Epoch {epoch+1}/{epochs}")
        print(f"Average training loss: {avg_train_loss:.4f}")
        print(f"Validation Accuracy: {val_accuracy:.4f}, Validation F1: {val_f1:.4f}")
        
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            torch.save(model.state_dict(), f'best_{model_name.lower()}_model.pt')
            print(f"Saved best {model_name} model")

# Evaluation function
def evaluate_model(model, data_loader, desc="Evaluation"):
    model.eval()
    predictions = []
    actual_labels = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc=desc):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            _, preds = torch.max(outputs.logits, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    
    accuracy = accuracy_score(actual_labels, predictions)
    f1 = f1_score(actual_labels, predictions, average='weighted', zero_division=0)
    return accuracy, f1

# Main execution
if __name__ == "__main__":
    # Create sample dataset
    create_sample_dataset()

    # Initialize tokenizers and models
    print("Loading models...")
    codebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
    bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    
    codebert_model = AutoModelForSequenceClassification.from_pretrained(
        "microsoft/codebert-base",
        num_labels=2
    ).to(device)
    
    bert_model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=2
    ).to(device)

    # Create datasets
    print("Creating datasets...")
    train_dataset_codebert = CodeDataset('data/train.jsonl', codebert_tokenizer)
    val_dataset_codebert = CodeDataset('data/valid.jsonl', codebert_tokenizer)
    test_dataset_codebert = CodeDataset('data/test.jsonl', codebert_tokenizer)

    train_dataset_bert = CodeDataset('data/train.jsonl', bert_tokenizer)
    val_dataset_bert = CodeDataset('data/valid.jsonl', bert_tokenizer)
    test_dataset_bert = CodeDataset('data/test.jsonl', bert_tokenizer)

    # Check if we have data
    if len(train_dataset_codebert) == 0 or len(train_dataset_bert) == 0:
        print("No data found. Exiting.")
        exit()

    # Create data loaders
    batch_size = 2  # Small batch size for our small dataset
    train_loader_codebert = DataLoader(train_dataset_codebert, batch_size=batch_size, shuffle=True)
    val_loader_codebert = DataLoader(val_dataset_codebert, batch_size=batch_size)
    test_loader_codebert = DataLoader(test_dataset_codebert, batch_size=batch_size)

    train_loader_bert = DataLoader(train_dataset_bert, batch_size=batch_size, shuffle=True)
    val_loader_bert = DataLoader(val_dataset_bert, batch_size=batch_size)
    test_loader_bert = DataLoader(test_dataset_bert, batch_size=batch_size)

    # Train models
    print("Training CodeBERT model...")
    train_model(codebert_model, train_loader_codebert, val_loader_codebert, "CodeBERT", epochs=3)

    print("\nTraining BERT model...")
    train_model(bert_model, train_loader_bert, val_loader_bert, "BERT", epochs=3)

    # Evaluate on test set
    print("\nEvaluating CodeBERT on test set...")
    codebert_accuracy, codebert_f1 = evaluate_model(codebert_model, test_loader_codebert, "CodeBERT Test")

    print("Evaluating BERT on test set...")
    bert_accuracy, bert_f1 = evaluate_model(bert_model, test_loader_bert, "BERT Test")

    # Comparative analysis
    print("\n=== Comparative Analysis ===")
    print(f"CodeBERT - Accuracy: {codebert_accuracy:.4f}, F1-score: {codebert_f1:.4f}")
    print(f"BERT - Accuracy: {bert_accuracy:.4f}, F1-score: {bert_f1:.4f}")

    if codebert_accuracy > bert_accuracy:
        print("CodeBERT outperforms BERT on accuracy")
    else:
        print("BERT outperforms CodeBERT on accuracy")
        
    if codebert_f1 > bert_f1:
        print("CodeBERT outperforms BERT on F1-score")
    else:
        print("BERT outperforms CodeBERT on F1-score")

    # Save results
    results = {
        'CodeBERT': {'Accuracy': codebert_accuracy, 'F1': codebert_f1},
        'BERT': {'Accuracy': bert_accuracy, 'F1': bert_f1}
    }
    
    with open('results.json', 'w') as f:
        json.dump(results, f, indent=4)
    
    print("\nResults saved to results.json")
    
    # Explanation of results
    print("\n=== Explanation ===")
    print("CodeBERT is pre-trained on both programming languages and natural language, so it should")
    print("perform better on code-related tasks like bug detection. BERT is only pre-trained on")
    print("natural language, so it may struggle with code syntax and semantics.")
    print("If CodeBERT performs better, it demonstrates the value of code-specific pre-training.")

Using device: cpu
Created sample dataset with 10 code examples
Loading models...
